In [1]:
import warnings, os
warnings.filterwarnings("ignore")

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from transformers import pipeline

2025-04-07 00:16:44.260438: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-07 00:16:44.271586: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-07 00:16:44.274771: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-07 00:16:44.283396: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
torch.cuda.empty_cache()
print(torch.cuda.is_available())   # True olmalı
print(torch.cuda.device_count())  # En az 1 olmalı
print(torch.cuda.get_device_name(0))  # GPU adını verir

True
1
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
model_id = "openai/whisper-small"
save_path = "../models/whisper-small"

required_files = ["model.safetensors", "config.json", "special_tokens_map.json"]
if not all(os.path.exists(os.path.join(save_path, file)) for file in required_files):
    print("Modelin bazı dosyaları eksik, indiriliyor...")
    # Modeli indir
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id)
    processor = AutoProcessor.from_pretrained(model_id)

    # DİKKAT: Hem model hem processor ayrı ayrı kaydedilmeli
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)

In [6]:
local_dir = "../models/whisper-small"

# Model ve processor'ı lokalden yükle
model = WhisperForConditionalGeneration.from_pretrained(local_dir)
processor = WhisperProcessor.from_pretrained(local_dir)

# tokenizer ve fe çalışmazsa aşağıdaki şekliyle kullanılabilir.
# from transformers import WhisperTokenizer, WhisperFeatureExtractor
# tokenizer = WhisperTokenizer.from_pretrained(local_dir)
# tokenizer.set_prefix_tokens(language="turkish")
# fe = WhisperFeatureExtractor.from_pretrained(local_dir)

tokenizer = processor.tokenizer
tokenizer.set_prefix_tokens(language="turkish")
fe = processor.feature_extractor

# Pipeline ile çalıştır (isteğe bağlı)
pipe = pipeline("automatic-speech-recognition",
                model=model,
                tokenizer=tokenizer,
                feature_extractor=fe,
                device=0 if torch.cuda.is_available() else -1,
                generate_kwargs={"language": "<|tr|>", "task": "transcribe"}
                )

# Ses dosyasını transkribe et
result = pipe("../../Downloads/hbdi.wav")

with open("hbdi.txt", "w") as file:
    file.write(result["text"])

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
